In [2]:
import torch
from torch import nn

In [3]:
import math
import tqdm

In [4]:
class LayerNorm(nn.Module):
  def __init__(self, d_model, eps=1e-12):
    super(LayerNorm, self).__init__()
    self.gamma = nn.Parameter(torch.ones(d_model))
    self.beta = nn.Parameter(torch.zeros(d_model))
    self.eps = eps

  def forward(self, x):
    mean = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    out = (x - mean) / torch.sqrt(var + self.eps)
    out = self.gamma * out + self.beta
    return out


In [5]:
#Multi head attention:
class MultiHeadAttention(nn.Module):
   def __init__(self,d_model,n_head):
     super(MultiHeadAttention,self).__init__()
     self.n_head=n_head
     self.attention=ScaleDotProductAttention()
     self.w_q=nn.Linear(d_model,d_model)
     self.w_k=nn.Linear(d_model,d_model)
     self.w_v=nn.Linear(d_model,d_model)
     self.w_concat=nn.Linear(d_model,d_model)

   def forward(self,q,k,v,mask=None):
     q,k,v=self.w_q(q),self.w_k(k),self.w_v(v)
     q,k,v=self.split(q),self.split(k),self.split(v)
     out,attention=self.attention(q,k,v,mask=mask)
     out=self.concat(out)
     out=self.w_concat(out)
     return out

   def split(self,tensor):
    batch_size,length,d_model=tensor.size()
    d_tensor=d_model//self.n_head
    tensor=tensor.view(batch_size,length,self.n_head,d_tensor).transpose(1,2)
    return tensor

   def concat(self,tensor):
    batch_size,head,length,d_tensor=tensor.size()
    d_model=head*d_tensor
    tensor=tensor.transpose(1,2).contiguous().view(batch_size,length,d_model)
    return tensor






In [6]:
#position_wise_feed_forward
class PositionWiseFeedForward(nn.Module):
  def __init__(self, d_model, hidden, drop_prob=0.1):
    super(PositionWiseFeedForward, self).__init__()
    self.linear1 = nn.Linear(d_model, hidden)
    self.linear2 = nn.Linear(hidden, d_model)
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(p=drop_prob)

  def forward(self, x):
    x = self.linear1(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.linear2(x)
    return x



In [7]:
class ScaleDotProductAttention(nn.Module):
  def __init__(self):
    super(ScaleDotProductAttention, self).__init__()
    self.softmax = nn.Softmax(dim=-1)

  def forward(self, q, k, v, mask=None, e=1e-12):
    batch_size, head, length, d_tensor = k.size()
    k_t = k.transpose(2, 3)
    score = (q @ k_t) / math.sqrt(d_tensor)

    if mask is not None:
      score = score.masked_fill(mask == 0, -10000)
    score = self.softmax(score)
    v = score @ v
    return v, score

In [8]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len, device):
    super(PositionalEncoding, self).__init__()
    self.encoding = torch.zeros(max_len, d_model, device=device)
    self.encoding.requires_grad = False

    pos = torch.arange(0, max_len, device=device).float().unsqueeze(dim=1)
    _2i = torch.arange(0, d_model, step=2, device=device).float()

    self.encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))
    self.encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))

  def forward(self, x):
    batch_size, seq_len = x.size()
    return self.encoding[:seq_len, :]

In [9]:
#tokenembedding
class TokenEmbedding(nn.Embedding):
  def __init__(self,vocab_size,d_model):
    super(TokenEmbedding,self).__init__(vocab_size,d_model,padding_idx=1)

In [10]:
class TransformerEmbedding(nn.Module):
  def __init__(self,vocab_size,d_model,max_len,drop_prob,device):
    super(TransformerEmbedding,self).__init__()
    self.tok_emb=TokenEmbedding(vocab_size,d_model)
    self.pos_emb=PositionalEncoding(d_model,max_len,device)
    self.drop_out=nn.Dropout(p=drop_prob)
  def forward(self,x):
    tok_emb=self.tok_emb(x)
    pos_emb=self.pos_emb(x)
    return self.drop_out(tok_emb+pos_emb)

In [11]:
class EncoderLayer(nn.Module):
  def __init__(self, d_model, ffn_hidden, n_head, drop_prob):
    super(EncoderLayer, self).__init__()
    self.attention = MultiHeadAttention(d_model=d_model, n_head=n_head)
    self.norm1 = LayerNorm(d_model=d_model)
    self.dropout1 = nn.Dropout(p=drop_prob)
    self.ffn = PositionWiseFeedForward(d_model=d_model, hidden=ffn_hidden, drop_prob=drop_prob)
    self.norm2 = LayerNorm(d_model=d_model)
    self.dropout2 = nn.Dropout(p=drop_prob)

  def forward(self, x, src_mask):
    _x = x
    x = self.attention(q=x, k=x, v=x, mask=src_mask)
    x = self.dropout1(x)
    x = self.norm1(x + _x)
    _x = x
    x = self.ffn(x)
    x = self.dropout2(x)
    x = self.norm2(x + _x)
    return x

In [12]:
class DecoderLayer(nn.Module):
  def __init__(self, d_model, ffn_hidden, n_head, drop_prob):
    super(DecoderLayer, self).__init__()
    self.self_attention = MultiHeadAttention(d_model=d_model, n_head=n_head)
    self.norm1 = LayerNorm(d_model=d_model)
    self.dropout1 = nn.Dropout(p=drop_prob)

    self.enc_dec_attention = MultiHeadAttention(d_model=d_model, n_head=n_head)
    self.norm2 = LayerNorm(d_model=d_model)
    self.dropout2 = nn.Dropout(p=drop_prob)

    self.ffn = PositionWiseFeedForward(d_model=d_model, hidden=ffn_hidden, drop_prob=drop_prob)
    self.norm3 = LayerNorm(d_model=d_model)
    self.dropout3 = nn.Dropout(p=drop_prob)

  def forward(self, dec, enc, trg_mask, src_mask):
    _x = dec
    x = self.self_attention(q=dec, k=dec, v=dec, mask=trg_mask)
    x = self.dropout1(x)
    x = self.norm1(x + _x)
    if enc is not None:
      _x = x
      x = self.enc_dec_attention(q=x, k=enc, v=enc, mask=src_mask)
      x = self.dropout2(x)
      x = self.norm2(x + _x)
    _x = x
    x = self.ffn(x)
    x = self.dropout3(x)
    x = self.norm3(x + _x)
    return x



In [13]:
class Encoder(nn.Module):
  def __init__(self, enc_voc_size, max_len, d_model, ffn_hidden, n_head, n_layers, drop_prob, device):
    super().__init__()
    self.emb = TransformerEmbedding(d_model=d_model, max_len=max_len, vocab_size=enc_voc_size, drop_prob=drop_prob, device=device)
    self.layers = nn.ModuleList([EncoderLayer(d_model=d_model, ffn_hidden=ffn_hidden, n_head=n_head, drop_prob=drop_prob) for _ in range(n_layers)])

  def forward(self, x, src_mask):
    x = self.emb(x)
    for layer in self.layers:
      x = layer(x, src_mask)
    return x

In [14]:
class Decoder(nn.Module):
    def __init__(self, dec_voc_size, max_len, d_model, ffn_hidden, n_head, n_layers, drop_prob, device):
        super().__init__()
        self.emb = TransformerEmbedding(d_model=d_model,
                                        drop_prob=drop_prob,
                                        max_len=max_len,
                                        vocab_size=dec_voc_size,
                                        device=device)

        self.layers = nn.ModuleList([DecoderLayer(d_model=d_model,ffn_hidden=ffn_hidden,
                                                  n_head=n_head,
                                                  drop_prob=drop_prob)
                                     for _ in range(n_layers)])

        self.linear = nn.Linear(d_model, dec_voc_size)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        trg = self.emb(trg)

        for layer in self.layers:
            trg = layer(trg, enc_src, trg_mask, src_mask)
        output = self.linear(trg)
        return output

In [15]:
class Transformer(nn.Module):

    def __init__(self, src_pad_idx, trg_pad_idx, trg_sos_idx, enc_voc_size, dec_voc_size, d_model, n_head, max_len,
                 ffn_hidden, n_layers, drop_prob, device):
        super().__init__()
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.trg_sos_idx = trg_sos_idx
        self.device = device
        self.encoder = Encoder(d_model=d_model,
                               n_head=n_head,
                               max_len=max_len,
                               ffn_hidden=ffn_hidden,
                               enc_voc_size=enc_voc_size,
                               drop_prob=drop_prob,
                               n_layers=n_layers,
                               device=device)

        self.decoder = Decoder(d_model=d_model,
                               n_head=n_head,
                               max_len=max_len,
                               ffn_hidden=ffn_hidden,
                               dec_voc_size=dec_voc_size,
                               drop_prob=drop_prob,
                               n_layers=n_layers,
                               device=device)

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_src = self.encoder(src, src_mask)
        output = self.decoder(trg, enc_src, trg_mask, src_mask)
        return output

    def make_src_mask(self, src):
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask

    def make_trg_mask(self, trg):
        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(3)
        trg_len = trg.shape[1]
        trg_sub_mask = torch.tril(torch.ones(trg_len, trg_len)).type(torch.ByteTensor).to(self.device)
        trg_mask = trg_pad_mask & trg_sub_mask
        return trg_mask

In [16]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

zip_path = '/content/drive/MyDrive/english_to_hindi_traslation_dataset.zip'
extract_path = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done! Files:")
print(os.listdir(extract_path))

Mounted at /content/drive
Done! Files:
['hindi to english dataset']


In [17]:
import os

for root, dirs, files in os.walk('/content/dataset'):
    for file in files:
        print(os.path.join(root, file))

/content/dataset/hindi to english dataset/opus.en-hi-train.hi
/content/dataset/hindi to english dataset/opus.en-hi-dev.en
/content/dataset/hindi to english dataset/opus.en-hi-train.en
/content/dataset/hindi to english dataset/opus.en-hi-dev.hi
/content/dataset/hindi to english dataset/opus.en-hi-test.hi
/content/dataset/hindi to english dataset/opus.en-hi-test.en


In [18]:
BASE = '/content/dataset/hindi to english dataset'

def load_file(path):
    with open(path, encoding='utf-8') as f:
        return [line.strip() for line in f.readlines()]

en_train = load_file(f'{BASE}/opus.en-hi-train.en')
hi_train = load_file(f'{BASE}/opus.en-hi-train.hi')
en_dev = load_file(f'{BASE}/opus.en-hi-dev.en')
hi_dev = load_file(f'{BASE}/opus.en-hi-dev.hi')
en_test = load_file(f'{BASE}/opus.en-hi-test.en')
hi_test = load_file(f'{BASE}/opus.en-hi-test.hi')

print(f"Train pairs: {len(en_train)}")
print(f"Dev pairs:   {len(en_dev)}")
print(f"Test pairs:  {len(en_test)}")

print("\nSample pair:")
print("EN:", en_train[0])
print("HI:", hi_train[0])

Train pairs: 534319
Dev pairs:   2000
Test pairs:  2000

Sample pair:
EN: Other, Private Use
HI: अन्य, निज़ी उपयोग


In [19]:
MAX_TRAIN_LEN = 100

filtered = [(en, hi) for en, hi in zip(en_train, hi_train)
            if len(en.split()) <= MAX_TRAIN_LEN
            and len(hi.split()) <= MAX_TRAIN_LEN]

en_train, hi_train = zip(*filtered)
print(f"Filtered: {len(filtered)} pairs remaining")

Filtered: 531281 pairs remaining


In [20]:
!pip install sentencepiece -q

In [21]:
import sentencepiece as spm
with open('/content/combined.txt', 'w', encoding='utf-8') as f:
    for en, hi in zip(en_train, hi_train):
        f.write(en + '\n')
        f.write(hi + '\n')

In [22]:
spm.SentencePieceTrainer.train(
    input='/content/combined.txt',
    model_prefix='/content/en_hi_spm',
    vocab_size=16000,
    character_coverage=0.9995,
    model_type='bpe',
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece='<pad>',
    unk_piece='<unk>',
    bos_piece='<sos>',
    eos_piece='<eos>'
)

print("Tokenizer trained!")

sp = spm.SentencePieceProcessor()
sp.load('/content/en_hi_spm.model')

print("\nEnglish tokenized:", sp.encode(en_train[0], out_type=str))
print("Hindi tokenized:  ", sp.encode(hi_train[0], out_type=str))
print("\nVocab size:", sp.get_piece_size())

Tokenizer trained!

English tokenized: ['▁Other', ',', '▁Pri', 'v', 'ate', '▁Use']
Hindi tokenized:   ['▁अन्य', ',', '▁नि', 'ज़ी', '▁उपयोग']

Vocab size: 16000


In [23]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

PAD_IDX = sp.piece_to_id('<pad>')
SOS_IDX = sp.piece_to_id('<sos>')
EOS_IDX = sp.piece_to_id('<eos>')
UNK_IDX = sp.piece_to_id('<unk>')

print(f"PAD={PAD_IDX}, SOS={SOS_IDX}, EOS={EOS_IDX}, UNK={UNK_IDX}")

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, trg_sentences):
        self.src = src_sentences
        self.trg = trg_sentences

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src = [SOS_IDX] + sp.encode(self.src[idx]) + [EOS_IDX]
        trg = [SOS_IDX] + sp.encode(self.trg[idx]) + [EOS_IDX]
        return torch.tensor(src, dtype=torch.long), torch.tensor(trg, dtype=torch.long)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=PAD_IDX, batch_first=True)
    return src_batch, trg_batch

train_dataset = TranslationDataset(en_train, hi_train)
dev_dataset   = TranslationDataset(en_dev,   hi_dev)
test_dataset  = TranslationDataset(en_test,  hi_test)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

src_batch, trg_batch = next(iter(train_loader))
print(f"\nSource batch shape: {src_batch.shape}")
print(f"Target batch shape: {trg_batch.shape}")
print(f"\nSample source tokens: {src_batch[0][:10]}")
print(f"Sample target tokens: {trg_batch[0][:10]}")

PAD=0, SOS=2, EOS=3, UNK=1

Source batch shape: torch.Size([32, 82])
Target batch shape: torch.Size([32, 91])

Sample source tokens: tensor([   2, 9578,   60, 1907, 3322,   52, 6592, 2978,    3,    0])
Sample target tokens: tensor([   2, 6064,   57, 9198, 7608, 3317,   78,  903, 4736,  555])


In [24]:
import torch
import torch.nn as nn

VOCAB_SIZE = sp.get_piece_size()
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 4
D_FF = 512
DROPOUT = 0.1
MAX_LEN = 900
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")
print(f"Vocab size: {VOCAB_SIZE}")

model = Transformer(
    src_pad_idx=PAD_IDX,
    trg_pad_idx=PAD_IDX,
    trg_sos_idx=SOS_IDX,
    enc_voc_size=VOCAB_SIZE,
    dec_voc_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_head=N_HEADS,
    max_len=MAX_LEN,
    ffn_hidden=D_FF,
    n_layers=N_LAYERS,
    drop_prob=DROPOUT,
    device=DEVICE
).to(DEVICE)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Model parameters: {count_parameters(model):,}")
print("Model ready!")

Device: cuda
Vocab size: 16000
Model parameters: 17,575,552
Model ready!


In [25]:
print(type(model))

<class '__main__.Transformer'>


In [26]:

BATCH_SIZE = 32

en_train_small = en_train[:50000]
hi_train_small = hi_train[:50000]

train_dataset_small = TranslationDataset(en_train_small, hi_train_small)
train_loader_small  = DataLoader(train_dataset_small, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
train_loader_full   = DataLoader(train_dataset,       batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dev_loader          = DataLoader(dev_dataset,         batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader         = DataLoader(test_dataset,        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Small train batches : {len(train_loader_small)}")
print(f"Full train batches  : {len(train_loader_full)}")
print(f"Dev batches         : {len(dev_loader)}")

Small train batches : 1563
Full train batches  : 16603
Dev batches         : 63


In [27]:

import math
import time

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)

class WarmupScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=2000):
        self.optimizer    = optimizer
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        self.current_step = 0

    def step(self):
        self.current_step += 1
        lr = (self.d_model ** -0.5) * min(
            self.current_step ** -0.5,
            self.current_step * self.warmup_steps ** -1.5
        )
        for p in self.optimizer.param_groups:
            p['lr'] = lr

optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler = WarmupScheduler(optimizer, d_model=D_MODEL)

In [28]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [29]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc="Training", leave=False)
    for src, trg in loop:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        trg_input  = trg[:, :-1]
        trg_output = trg[:, 1:].contiguous().view(-1)

        optimizer.zero_grad()
        output = model(src, trg_input)
        output = output.contiguous().view(-1, output.shape[-1])

        loss = criterion(output, trg_output)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scheduler.step()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    loop = tqdm(loader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for src, trg in loop:
            src, trg = src.to(DEVICE), trg.to(DEVICE)
            trg_input  = trg[:, :-1]
            trg_output = trg[:, 1:].contiguous().view(-1)
            output = model(src, trg_input)
            output = output.contiguous().view(-1, output.shape[-1])
            loss   = criterion(output, trg_output)
            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)

In [30]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.stop       = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            print(f"  → EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stop = True

In [31]:
from tqdm import tqdm

In [ ]:
optimizer      = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler      = WarmupScheduler(optimizer, d_model=D_MODEL, warmup_steps=2000)
criterion      = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
best_val_loss  = float('inf')
early_stopping = EarlyStopping(patience=5, min_delta=0.001)

print("=== Full Training ===\n")
for epoch in range(30):
    start      = time.time()
    train_loss = train_epoch(model, train_loader_full, optimizer, criterion)
    val_loss   = evaluate(model, dev_loader, criterion)
    elapsed    = time.time() - start

    print(f"Epoch {epoch+1}/20 | Time: {elapsed:.1f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model_v2.pt')
        print(f"  → Saved! Val Loss: {val_loss:.4f}")

    early_stopping(val_loss)
    if early_stopping.stop:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nBest Val Loss: {best_val_loss:.4f}")

=== Full Training ===



Epoch 1/20 | Time: 1459.9s | Train Loss: 4.6578 | Val Loss: 4.7627
  → Saved! Val Loss: 4.7627


Epoch 2/20 | Time: 1455.8s | Train Loss: 3.7871 | Val Loss: 4.6804
  → Saved! Val Loss: 4.6804


Epoch 3/20 | Time: 1453.8s | Train Loss: 3.5232 | Val Loss: 4.6640
  → Saved! Val Loss: 4.6640


Epoch 4/20 | Time: 1454.2s | Train Loss: 3.3640 | Val Loss: 4.6427
  → Saved! Val Loss: 4.6427


Epoch 5/20 | Time: 1447.8s | Train Loss: 3.2482 | Val Loss: 4.6378
  → Saved! Val Loss: 4.6378


Training:  21%|██        | 3507/16603 [05:02<16:46, 13.01it/s, loss=3.2] 

In [32]:
import os

checkpoint_path = '/content/drive/MyDrive/best_model_v2.pt'

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    print(f"Model loaded from {checkpoint_path}")
else:
    print("No checkpoint found — starting from scratch")

model = model.to(DEVICE)

optimizer      = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler      = WarmupScheduler(optimizer, d_model=D_MODEL, warmup_steps=2000)
criterion      = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
best_val_loss  = float('inf')
early_stopping = EarlyStopping(patience=5, min_delta=0.001)

print("=== Resuming Full Training ===\n")
for epoch in range(30):
    start      = time.time()
    train_loss = train_epoch(model, train_loader_full, optimizer, criterion)
    val_loss   = evaluate(model, dev_loader, criterion)
    elapsed    = time.time() - start

    print(f"Epoch {epoch+1}/30 | Time: {elapsed:.1f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  → Saved! Val Loss: {val_loss:.4f}")

    early_stopping(val_loss)
    if early_stopping.stop:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nBest Val Loss: {best_val_loss:.4f}")

Model loaded from /content/drive/MyDrive/best_model_v2.pt
=== Resuming Full Training ===



Epoch 1/30 | Time: 1473.0s | Train Loss: 3.6765 | Val Loss: 4.6859
  → Saved! Val Loss: 4.6859


Epoch 2/30 | Time: 1476.1s | Train Loss: 3.3024 | Val Loss: 4.6306
  → Saved! Val Loss: 4.6306


KeyboardInterrupt: 

In [33]:
def translate(sentence, model, sp, device, max_len=50):
    model.eval()
    tokens = [SOS_IDX] + sp.encode(sentence) + [EOS_IDX]
    src = torch.tensor(tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        src_mask = model.make_src_mask(src)
        enc_src = model.encoder(src, src_mask)

    trg_tokens = [SOS_IDX]
    for _ in range(max_len):
        trg = torch.tensor(trg_tokens).unsqueeze(0).to(device)
        with torch.no_grad():
            trg_mask = model.make_trg_mask(trg)
            output = model.decoder(trg, enc_src, trg_mask, src_mask)
        next_token = output.argmax(-1)[:, -1].item()
        trg_tokens.append(next_token)
        if next_token == EOS_IDX:
            break

    translated = sp.decode(trg_tokens[1:-1])
    return translated

def check_in_vocab(sentence, sp):
    pieces = sp.encode(sentence, out_type=str)
    oov = [p for p in pieces if p.startswith('<unk>') or p == '<unk>']
    return pieces, len(oov) == 0

test_sentences = [
    "Hello, how are you?",
    "I love machine learning.",
    "The weather is nice today.",
    "My name is Stephen.",
    "The government announced a new policy.",
    "She is going to school.",
]

for sentence in test_sentences:
    pieces, in_vocab = check_in_vocab(sentence, sp)
    if in_vocab:
        translation = translate(sentence, model, sp, DEVICE)
        print(f"EN: {sentence}")
        print(f"HI: {translation}")
    else:
        print(f"EN: {sentence}")
        print(f"⚠ Skipped — OOV tokens: {[p for p in pieces if '▁' not in p and len(p) <= 2]}")
    print()

EN: Hello, how are you?
HI: नमस्कार, तुम कैसे हो?

EN: I love machine learning.
HI: मैं सबक से प्रेम करता हूँ.

EN: The weather is nice today.
HI: मौसम आज अच्छा है.

EN: My name is Stephen.
HI: मेरा नाम स्टीफन है.

EN: The government announced a new policy.
HI: सरकार ने एक नई नीति की नीति को दिया.

EN: She is going to school.
HI: वह स्कूल स्कूल जा रहा है.



In [34]:
!pip install sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00


In [35]:
from sacrebleu.metrics import BLEU

def evaluate_bleu(model, test_en, test_hi, sp, device, n=500):
    hypotheses = []
    references = []

    for en, hi in zip(test_en[:n], test_hi[:n]):
        pred = translate(en, model, sp, device)
        hypotheses.append(pred)
        references.append(hi)

    bleu = BLEU()
    score = bleu.corpus_score(hypotheses, [references])
    return score

# Load best model first
model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_v2.pt'))
model.eval()

score = evaluate_bleu(model, en_test, hi_test, sp, DEVICE)
print(f"BLEU Score: {score}")

BLEU Score: BLEU = 11.53 41.9/17.2/8.9/5.3 (BP = 0.849 ratio = 0.859 hyp_len = 6390 ref_len = 7439)
